In [3]:
import asyncio, json, random, time, threading
from pathlib import Path
from datetime import datetime
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

In [4]:
# Jobstreet MY — Clean Web Scraper
# Crawls specific fields -> data/raw_datas.json (JSONL format)

TARGET, MAX_PAGES, RETRIES = 105_000, 1000, 3
CONCURRENCY = 8  # max parallel browser tabs
OUT = Path("data/raw_datas.json")
OUT.parent.mkdir(parents=True, exist_ok=True)

CLASS_IDS = list(range(1, 31))
total = 0
counts = {c: 0 for c in CLASS_IDS}
log = lambda msg: print(f"[{datetime.now():%H:%M:%S}] {msg}", flush=True)

async def scrape_page(page, cls_id, pg):
    global total
    url = f"https://my.jobstreet.com/jobs?classification={cls_id}&page={pg}"

    for attempt in range(1, RETRIES + 1):
        try:
            await page.goto(url, wait_until="commit", timeout=30_000)
            await page.wait_for_timeout(random.randint(2000, 4000))
            try:
                await page.wait_for_selector("article", timeout=8_000)
            except Exception:
                pass
            html = await page.content()
            break
        except Exception as e:
            if attempt == RETRIES:
                log(f"[CLS-{cls_id}] SKIP pg {pg} after {RETRIES} retries -- {e}")
                return 0
            await asyncio.sleep(random.uniform(3, 6))

    articles = BeautifulSoup(html, "html.parser").find_all("article")
    if not articles:
        return 0

    records = []
    for a in articles:
        def get_txt(qa):
            el = a.find(attrs={"data-automation": qa})
            return el.get_text(strip=True) if el else ""
        records.append({
            "job_title": get_txt("jobTitle"),
            "company": get_txt("jobCompany"),
            "location": get_txt("jobLocation"),
            "classification": get_txt("jobClassification"),
            "salary": get_txt("jobSalary")
        })

    rows = "\n".join(json.dumps(r, ensure_ascii=False) for r in records) + "\n"
    n = len(articles)

    async with flock:
        with open(OUT, "a", encoding="utf-8") as f:
            f.write(rows)
    return n

async def worker(browser, cls_id, sem):
    global total
    async with sem:
        ctx = await browser.new_context(
            viewport={"width": 1280, "height": 800},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
        )
        tab = await ctx.new_page()

        for pg in range(1, MAX_PAGES + 1):
            if total >= TARGET:
                break

            n = await scrape_page(tab, cls_id, pg)
            if n == 0:
                break

            counts[cls_id] += n
            total += n
            log(f"[CLS-{cls_id}] pg {pg} (+{n}).  Cat: {counts[cls_id]:,}  |  Total: {total:,} / {TARGET:,}")
            await asyncio.sleep(random.uniform(1.5, 3.5))

        await tab.close()
        await ctx.close()
        log(f"[CLS-{cls_id}] Worker done -- {counts[cls_id]:,} cards.")

async def main():
    global flock, total
    flock = asyncio.Lock()
    log(f"Output -> {OUT.resolve()}")
    log(f"Launching {len(CLASS_IDS)} classification workers (max {CONCURRENCY} concurrent)...")
    t0 = time.perf_counter()

    sem = asyncio.Semaphore(CONCURRENCY)

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(
            headless=True,
            args=["--disable-blink-features=AutomationControlled", "--disable-dev-shm-usage", "--no-sandbox"],
        )
        await asyncio.gather(*(worker(browser, c, sem) for c in CLASS_IDS))
        await browser.close()

    mins = (time.perf_counter() - t0) / 60
    log("=" * 60)
    log(f"DONE -- {total:,} cards in {mins:.1f} min")
    for c in CLASS_IDS:
        if counts[c] > 0:
            log(f"  Classification {c:>2}: {counts[c]:>7,}")
    log(f"File: {OUT.resolve()}  ({OUT.stat().st_size / 1048576:.1f} MB)")

def _run_crawler():
    loop = asyncio.ProactorEventLoop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(main())
    loop.close()

t = threading.Thread(target=_run_crawler)
t.start()
t.join()

[19:59:16] Output -> C:\Users\Polok\OneDrive - Universiti Teknologi Malaysia (UTM)\Documents\Y3S2\High performance Data Processing\Project\data\raw_datas.json
[19:59:16] Launching 30 classification workers (max 8 concurrent)...
[19:59:32] [CLS-1] pg 1 (+30).  Cat: 30  |  Total: 30 / 105,000
[19:59:35] [CLS-2] pg 1 (+30).  Cat: 30  |  Total: 60 / 105,000
[19:59:37] [CLS-3] pg 1 (+30).  Cat: 30  |  Total: 90 / 105,000
[19:59:38] [CLS-4] pg 1 (+30).  Cat: 30  |  Total: 120 / 105,000
[19:59:39] [CLS-1] pg 2 (+30).  Cat: 60  |  Total: 150 / 105,000
[19:59:41] [CLS-5] pg 1 (+30).  Cat: 30  |  Total: 180 / 105,000
[19:59:41] [CLS-2] pg 2 (+30).  Cat: 60  |  Total: 210 / 105,000
[19:59:42] [CLS-6] pg 1 (+30).  Cat: 30  |  Total: 240 / 105,000
[19:59:44] [CLS-7] pg 1 (+30).  Cat: 30  |  Total: 270 / 105,000
[19:59:45] [CLS-3] pg 2 (+30).  Cat: 60  |  Total: 300 / 105,000
[19:59:45] [CLS-4] pg 2 (+30).  Cat: 60  |  Total: 330 / 105,000
[19:59:47] [CLS-1] pg 3 (+30).  Cat: 90  |  Total: 360 / 105